# Prompt Evaluation - MODULE 2

## Setup

In [ ]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

## 1. Generating test datasets

In [11]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [39]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

### Testing the Dataset Generation

In [40]:
dataset = generate_dataset()
print(dataset)

[{'task': "Parse an AWS S3 object key to extract the bucket name, folder path, and file name from a full S3 URI like 's3://my-bucket/folder/subfolder/file.txt'", 'format': 'regex'}, {'task': 'Create a CloudFormation template JSON object that defines an S3 bucket with versioning enabled and public access blocked', 'format': 'json'}, {'task': "Write a Python function that takes an AWS IAM policy document as a dictionary and returns a list of all the resources that have 'Allow' statements", 'format': 'python'}]


### Saving the Dataset

In [41]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

## 2. Running the eval

The evaluation process follows a clear workflow: we take our dataset of test cases, combine each one with our prompt template, send it to Claude for processing, and then evaluate the output using a grader system.

### Building the Core Functions

The evaluation pipeline consists of three main functions, each with a specific responsibility. Let's start with the simplest one - the function that handles individual prompts

### The run_prompt Function

This function takes a test case and merges it with our prompt template:

In [16]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

### The run_test_case Function

This function orchestrates running a single test case and grading the result:

In [17]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

For now, we're using a hardcoded score of 10. The grading logic is where we'll spend significant time in upcoming sections, but this placeholder lets us test the overall pipeline.

### The run_eval Function

This function coordinates the entire evaluation process:

In [18]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

This function processes every test case in our dataset and collects all the results into a single list.

### Running the Evaluation

To execute our evaluation pipeline, we load our dataset and run it through our functions:

In [19]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

#### Examining the Results

In [20]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution: Extract AWS Account ID from ARN\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    AWS ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Args:\n        arn (str): The Amazon Resource Name (ARN) string\n        \n    Returns:\n        str: The 12-digit AWS account ID\n        \n    Raises:\n        ValueError: If the ARN format is invalid or account ID is not found\n    \"\"\"\n    try:\n        # Split ARN by colon delimiter\n        parts = arn.split(':')\n        \n        # ARN should have at least 6 parts: arn, partition, service, region, account-id, resource\n        if len(parts) < 6:\n            raise ValueError(f\"Invalid ARN format: {arn}\")\n        \n        # Account ID is the 5th element (index 4)\n        account_id = parts[4]\n        \n        # Validate that account ID is 12 digits\n        if not account_id.i

## 3. Model based grading

### Implementing a Model Grader

In [32]:
def grade_by_model(test_case, output, max_retries=3):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    for _ in range(max_retries):
        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(messages, stop_sequences=["```"])
        try:
            grade = json.loads(eval_text)
        except json.JSONDecodeError:
            continue
        if "score" in grade and "reasoning" in grade:
            return grade
    raise ValueError(f"Grader sem 'score' após {max_retries} tentativas. Última resposta: {eval_text}")


### Running the eval functions again, but now using the grading function

In [29]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [49]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score across all test cases: {average_score:.2f}")
    
    return results

In [52]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))


Average score across all test cases: 7.83
[
  {
    "output": "\nimport re\nfrom urllib.parse import urlparse\n\ndef parse_s3_uri(uri):\n    # Using regex approach\n    pattern = r's3://([^/]+)(?:/(.+)/)?([^/]+)$'\n    match = re.match(pattern, uri)\n    \n    if match:\n        bucket = match.group(1)\n        folder_path = match.group(2) or \"\"\n        file_name = match.group(3)\n        return {\n            \"bucket\": bucket,\n            \"folder_path\": folder_path,\n            \"file_name\": file_name\n        }\n    \n    return None\n\n# Test\nprint(parse_s3_uri('s3://my-bucket/folder/subfolder/file.txt'))\n",
    "test_case": {
      "task": "Parse an AWS S3 object key to extract the bucket name, folder path, and file name from a full S3 URI like 's3://my-bucket/folder/subfolder/file.txt'",
      "format": "regex"
    },
    "score": 6.5,
    "reasoning": "While the solution demonstrates basic problem-solving, the regex pattern `r's3://([^/]+)(?:/(.+)/)?([^/]+)$'` is fund

## 4. Code based grading

Note: We have updated the dataset generator cell to add "format".

### Validate code functions

In [53]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response) 
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)


Update the run_prompt function:

In [58]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

*Respond only with Python, Json ou a plain regex
*Do not add any comments, commentary ou explanations
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

Update the run_test_case function to use the new grade_syntax function:

In [59]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [60]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score across all test cases: {average_score:.2f}")
    
    return results

Run the eval function again:

In [61]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score across all test cases: 6.33
